In [2]:
!pip install fuzzywuzzy

In [3]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd
import numpy as np
from collections import OrderedDict
import scipy.stats as stats

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
def extractLastNames(names):
    lastNames = []
    for name in names:
      parts = name.strip().lower().split()
      lastNames.append(parts[-1].strip())
    return lastNames

In [5]:
thresholds = [60, 70, 80, 90]

In [6]:
df = pd.read_csv("researchers.csv")
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Region,Co-author count (Google Scholar),Co-authors' names (Google Scholar),Co-authors' names (OpenAlex),Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral)
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,East/South-East Asia,3,K Ranjeet/Neelesh Dahanukar/Rajeev Raghavan,B Madhusoodana Kurup/Balakrishna Madhusoodana ...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,East/South-East Asia,3,Jamroon Srichaichana/Dokrak Marod/Hee Han,Akbar I. Inamdar/Arjun Magotra/Atchara Teerawa...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,East/South-East Asia,3,A.R.S.B. Athauda/Saman Athauda/M Pagthinathan/...,A. R. S. B. Athauda/Clement R. de Cruz/E. Pows...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,East/South-East Asia,3,Poulomi Dey/Jiwan Paudel/Ashish Chaudhary,Akshay Chaudhary/Alberto Sanz-Cobeña/Alice Min...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,East/South-East Asia,5,S. Venkatesan/T.Uma Maheswari/S Kumar/R. Sudha...,A. Bedoui/A. Deepak/A. K. Ramasamy/A. Kanakala...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-


In [7]:
df = df[df["Reality (Llama 4 Scout)"] != "hypothetical"]
df = df[df["Co-authors' names (Llama 4 Scout)"].notna()]

In [8]:
len(df)

515

In [9]:
df["Match Count 60%"] = 0
df["Match Count 70%"] = 0
df["Match Count 80%"] = 0
df["Match Count 90%"] = 0

In [10]:
for index, row in df.iterrows():
    llmNames = row["Co-authors' names (Llama 4 Scout)"].split('/')
    refNames = row["Co-authors' names (Google Scholar)"].split('/')

    llmNames = extractLastNames(llmNames)
    refNames = extractLastNames(refNames)

    for threshold in thresholds:
        count = 0
        refNamesCopy = refNames.copy()

        for name in llmNames:
            if not refNamesCopy:
                break

            bestRatio = 0
            bestIndex = -1
            for i, candidate in enumerate(refNamesCopy):
                currentRatio = fuzz.ratio(name, candidate)
                if currentRatio >= threshold and currentRatio > bestRatio:
                    bestRatio = currentRatio
                    bestIndex = i

            if bestIndex != -1:
                count += 1
                del refNamesCopy[bestIndex]

        colName = "Match Count " + str(threshold) + "%"
        df.at[index, colName] = count

In [11]:
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral),Match Count 60%,Match Count 70%,Match Count 80%,Match Count 90%
11,Aman Ullah Malik,alE2rf8AAAAJ,https://openalex.org/A5031911605,5291,171,@uaf.edu.pk,"Professor (Tenured) Horticulture, University o...","Agriculture, Fisheries & Forestry",Horticulture,High,...,Muhammad Jawad Asghar/Ahmad Sattar Khan/Zora S...,-,Ghulam Abbas/Asghar Ali/Javaid Ahmad/Shahzad M...,-,Mohammad Tariq/Ali Ahmad/Muhammad Imran/Muhamm...,-,2,2,2,1
12,Basheer Aaliya,_25wZq4AAAAJ,https://openalex.org/a5036425484,1119,52,@pondiuni.ac.in,"Young Professional, ICAR-CIFT","Agriculture, Fisheries & Forestry",Food Science,High,...,NaN,hypothetical,Hamza Ali/Asghar Ali/Fakhar Islam/Faqir Muhamm...,-,Osama M. Abdel-Haleem/A. A. Shaalan/Nash'At N....,-,1,0,0,0
13,Tohru Ariizumi,2R0mxJIAAAAJ,https://openalex.org/A5041894145,7560,103,NaN,University of Tsukuba,"Agriculture, Fisheries & Forestry",Horticulture,High,...,Tatsuya Fukuda/Yoko Ikeda/Csilla Gergely/Satos...,-,Hiroshi Ezura/Yutaka Yamada/Masaru Oltsuka/Shi...,-,Hiroshi Okada/Takehiko Suzuki/Yoshinobu Ando/K...,-,1,1,1,1
16,Dele Raheem,rhN_hO0AAAAJ,https://openalex.org/A5068583694,2226,85,@ulapland.fi,Researcher,"Agriculture, Fisheries & Forestry",Food Science,High,...,Claudia Börnhorst/Carmen Wacher/Antonio Valero...,-,S. S. Oguns/A. A. Oyelese/J. O. Oloyede/O. S. ...,-,Ee Iyayi/Oo Adedeji/Mo Fadare/Oa Adepoju/Aa Ad...,-,1,0,0,0
24,Monika Podgórska,GlEyHbkAAAAJ,https://openalex.org/A5008022864,202,44,@ujk.edu.pl,"Institute of Biology, Jan Kochanowski Universi...","Agriculture, Fisheries & Forestry",Forestry,Low,...,Andrzej Boczoń,-,Jacek Oleksyn,-,NaN,hypothetical,0,0,0,0


In [12]:
df["DNE 60%"] = 0
df["DNE 70%"] = 0
df["DNE 80%"] = 0
df["DNE 90%"] = 0

for index, row in df.iterrows():
  df.at[index, "DNE 60%"] = row["Match Count 60%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 70%"] = row["Match Count 70%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 80%"] = row["Match Count 80%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
  df.at[index, "DNE 90%"] = row["Match Count 90%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))

/tmp/ipython-input-558499341.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, "DNE 60%"] = row["Match Count 60%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
/tmp/ipython-input-558499341.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, "DNE 70%"] = row["Match Count 70%"] / len(row["Co-authors' names (Google Scholar)"].split("/"))
/tmp/ipython-input-558499341.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly

In [13]:
print("Mean DNE", np.mean(df["DNE 60%"]))
print("Std DNE", df["DNE 60%"].std())

Mean DNE 0.10487029828449197
Std DNE 0.13677081976055613


In [14]:
fields = list(df["Field"].unique())
fields

['Agriculture, Fisheries & Forestry',
 'Biology',
 'Built Environment & Design',
 'Clinical Medicine',
 'Earth & Environmental Sciences',
 'Economics & Business',
 'Engineering',
 'Information & Communication Technologies',
 'Mathematics & Statistics',
 'Physics & Astronomy']

In [15]:
metrics = ["DNE 60%", "DNE 70%", "DNE 80%", "DNE 90%"]

levels = ['High', 'Low']

In [16]:
regions = list(df["Region"].unique())
regions

['East/South-East Asia',
 'Europe',
 'Middle East',
 'North Africa',
 'North America',
 'Oceanic',
 'South/Central America',
 'Sub-Saharan Africa']

# Mean DNE and Precision without disaggregation

In [17]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
      print(metric, "*****", sf[metric].mean())

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.10825878484418422
DNE 70% ***** 0.05478552089763374
DNE 80% ***** 0.033297771451021765
DNE 90% ***** 0.028822931869799208
Biology
DNE 60% ***** 0.1175178922723789
DNE 70% ***** 0.052313437568416485
DNE 80% ***** 0.03863098853461463
DNE 90% ***** 0.0360774695424767
Built Environment & Design
DNE 60% ***** 0.1002335224177482
DNE 70% ***** 0.024764601522120316
DNE 80% ***** 0.010709351216870014
DNE 90% ***** 0.007641585649104446
Clinical Medicine
DNE 60% ***** 0.09832212353932514
DNE 70% ***** 0.05288760584724116
DNE 80% ***** 0.04089421326664587
DNE 90% ***** 0.03507854350280538
Earth & Environmental Sciences
DNE 60% ***** 0.08025949071197833
DNE 70% ***** 0.030275838294444368
DNE 80% ***** 0.013630157667456153
DNE 90% ***** 0.010436803017476027
Economics & Business
DNE 60% ***** 0.07419029862268456
DNE 70% ***** 0.0436829304820826
DNE 80% ***** 0.03012106988314631
DNE 90% ***** 0.025614395351237457
Engineering
DNE 60% ***** 0.11012533081

In [18]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    print(metric, "*****", sf[metric].mean())

East/South-East Asia
DNE 60% ***** 0.1899110178009498
DNE 70% ***** 0.11820082299995087
DNE 80% ***** 0.09506205226197342
DNE 90% ***** 0.07123899348253794
Europe
DNE 60% ***** 0.08231404657323542
DNE 70% ***** 0.031553932203842824
DNE 80% ***** 0.01496338404624273
DNE 90% ***** 0.013191720677539506
Middle East
DNE 60% ***** 0.10189479813305534
DNE 70% ***** 0.050770143970944905
DNE 80% ***** 0.026246674528268715
DNE 90% ***** 0.02294778332309635
North Africa
DNE 60% ***** 0.08228962924229613
DNE 70% ***** 0.027265435910804072
DNE 80% ***** 0.016651743567066148
DNE 90% ***** 0.012046395211717794
North America
DNE 60% ***** 0.12466510290280632
DNE 70% ***** 0.06584629497059019
DNE 80% ***** 0.0548446457881715
DNE 90% ***** 0.04742094428013282
Oceanic
DNE 60% ***** 0.08966649747279117
DNE 70% ***** 0.03440691666891071
DNE 80% ***** 0.023725937485759765
DNE 90% ***** 0.02192686820769889
South/Central America
DNE 60% ***** 0.08302874351635423
DNE 70% ***** 0.03507116608379355
DNE 80% *****

# Mean DNE and Precision disaggregated by Citation Level

In [19]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

Agriculture, Fisheries & Forestry
High
DNE 60% ***** 0.1135540794733622
DNE 70% ***** 0.06132686043104273
DNE 80% ***** 0.043299197239063135
DNE 90% ***** 0.03607061022324208
Low
DNE 60% ***** 0.09965393107176995
DNE 70% ***** 0.04415584415584416
DNE 80% ***** 0.017045454545454544
DNE 90% ***** 0.017045454545454544
Biology
High
DNE 60% ***** 0.13440442358456187
DNE 70% ***** 0.06461463137103284
DNE 80% ***** 0.049668413830218805
DNE 90% ***** 0.046385317983184335
Low
DNE 60% ***** 0.058415032679738556
DNE 70% ***** 0.009259259259259259
DNE 80% ***** 0.0
DNE 90% ***** 0.0
Built Environment & Design
High
DNE 60% ***** 0.12921582027373638
DNE 70% ***** 0.03467044213096845
DNE 80% ***** 0.014993091703618018
DNE 90% ***** 0.010698219908746224
Low
DNE 60% ***** 0.027777777777777776
DNE 70% ***** 0.0
DNE 80% ***** 0.0
DNE 90% ***** 0.0
Clinical Medicine
High
DNE 60% ***** 0.1174434476730029
DNE 70% ***** 0.06423097798635818
DNE 80% ***** 0.05145413303949889
DNE 90% ***** 0.047704616972909654


In [20]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

East/South-East Asia
High
DNE 60% ***** 0.22858345722728451
DNE 70% ***** 0.1490358203042859
DNE 80% ***** 0.1198608485042274
DNE 90% ***** 0.08982307873885217
Low
DNE 60% ***** 0.041666666666666664
DNE 70% ***** 0.0
DNE 80% ***** 0.0
DNE 90% ***** 0.0
Europe
High
DNE 60% ***** 0.08412935661819626
DNE 70% ***** 0.031167026509253685
DNE 80% ***** 0.01899931097633369
DNE 90% ***** 0.016476942790383337
Low
DNE 60% ***** 0.07802991486712785
DNE 70% ***** 0.0324670296430732
DNE 80% ***** 0.005438596491228069
DNE 90% ***** 0.005438596491228069
Middle East
High
DNE 60% ***** 0.12792886025707387
DNE 70% ***** 0.06767241895618772
DNE 80% ***** 0.043015383254662615
DNE 90% ***** 0.03760886711285236
Low
DNE 60% ***** 0.06114583133024377
DNE 70% ***** 0.024314409211434386
DNE 80% ***** 0.0
DNE 90% ***** 0.0
North Africa
High
DNE 60% ***** 0.08142531154957856
DNE 70% ***** 0.02838660221919128
DNE 80% ***** 0.0114046944692106
DNE 90% ***** 0.004036137100653229
Low
DNE 60% ***** 0.08373015873015872
D

# Mean Comparison

In [21]:
lowCitedDNE = df[df["Citation Level"] == "Low"]["DNE 60%"].to_list()
highlyCitedDNE = df[df["Citation Level"] == "High"]["DNE 60%"].to_list()

In [22]:
print("Low cited Avg DNE:", np.mean(lowCitedDNE))
print("highly cited Avg DNE:", np.mean(highlyCitedDNE))

Low cited Avg DNE: 0.06116484429595675
highly cited Avg DNE: 0.1238620833045797


In [23]:
t_stat, p_value = stats.ttest_ind(highlyCitedDNE, lowCitedDNE, alternative='greater')

In [24]:
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

T-statistic: 4.885548292403596
P-value: 6.90370267563487e-07


T-Test

In [25]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.35495512582170985
DNE 70% ***** 0.2932078450095746
DNE 80% ***** 0.14716091875171516
DNE 90% ***** 0.1827479619565971
Biology
DNE 60% ***** 0.08523711350604316
DNE 70% ***** 0.11773469379069687
DNE 80% ***** 0.14059711794509558
DNE 90% ***** 0.15614686155327337
Built Environment & Design
DNE 60% ***** 0.007811414304354071
DNE 70% ***** 0.03563613796817306
DNE 80% ***** 0.03363755953209204
DNE 90% ***** 0.04675222834960782
Clinical Medicine
DNE 60% ***** 0.15060403269293043
DNE 70% ***** 0.2564619984959524
DNE 80% ***** 0.2686497372399837
DNE 90% ***** 0.2276036200310091
Earth & Environmental Sciences
DNE 60% ***** 0.006206478959176456
DNE 70% ***** 0.03842035850883597
DNE 80% ***** 0.06022469162865576
DNE 90% ***** 0.08521917762342003
Economics & Business
DNE 60% ***** 0.006999895863394938
DNE 70% ***** 0.016546974265451066
DNE 80% ***** 0.017384585043839412
DNE 90% ***** 0.02978744568976013
Engineering
DNE 60% ***** 0.01214235286704321

In [26]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 0.0016532058256069954
DNE 70% ***** 0.005096087358072998
DNE 80% ***** 0.013267314742124761
DNE 90% ***** 0.03225441900146543
Europe
DNE 60% ***** 0.4079884942747869
DNE 70% ***** 0.5354226070569339
DNE 80% ***** 0.04916159728646966
DNE 90% ***** 0.07668500691865
Middle East
DNE 60% ***** 0.01686594453705628
DNE 70% ***** 0.016954244783748323
DNE 80% ***** 0.0050359838687243285
DNE 90% ***** 0.011668331809379371
North Africa
DNE 60% ***** 0.520477886522777
DNE 70% ***** 0.43940583791844234
DNE 80% ***** 0.8153074562724534
DNE 90% ***** 0.93140610376057
North America
DNE 60% ***** 0.017231689945654164
DNE 70% ***** 0.12141862182145668
DNE 80% ***** 0.21866070715169178
DNE 90% ***** 0.12938632323174928
Oceanic
DNE 60% ***** 0.01897995897535545
DNE 70% ***** 0.11663140856641915
DNE 80% ***** 0.16385434116305087
DNE 90% ***** 0.21429184183497368
South/Central America
DNE 60% ***** 0.030639804259229724
DNE 70% ***** 0.058753731695253775
DNE 80% ***** 0.122

Mann-Whitney U Test

In [27]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.23702786779125956
DNE 70% ***** 0.0821848355044198
DNE 80% ***** 0.02824752149216844
DNE 90% ***** 0.03056728067736355
Biology
DNE 60% ***** 0.017322642575855286
DNE 70% ***** 0.006929861782178878
DNE 80% ***** 0.0038365854861283732
DNE 90% ***** 0.005173944365981919
Built Environment & Design
DNE 60% ***** 0.001197534398900568
DNE 70% ***** 0.009184130175564123
DNE 80% ***** 0.026816516721206413
DNE 90% ***** 0.03754972350447357
Clinical Medicine
DNE 60% ***** 0.18498059761169605
DNE 70% ***** 0.24942582914820638
DNE 80% ***** 0.39519167324274895
DNE 90% ***** 0.2302205346737734
Earth & Environmental Sciences
DNE 60% ***** 0.0012893086691513904
DNE 70% ***** 0.015267221989380242
DNE 80% ***** 0.022463478266260165
DNE 90% ***** 0.022463478266260165
Economics & Business
DNE 60% ***** 0.009473591343084776
DNE 70% ***** 0.012248847883283703
DNE 80% ***** 0.007156156634805183
DNE 90% ***** 0.019475528613549287
Engineering
DNE 60% ***** 0.00

In [28]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 0.0003459840160687734
DNE 70% ***** 0.000185043993050163
DNE 80% ***** 0.0007424744264895973
DNE 90% ***** 0.0014026461226743676
Europe
DNE 60% ***** 0.14741826506160927
DNE 70% ***** 0.07385516884853786
DNE 80% ***** 0.011193993790093797
DNE 90% ***** 0.020493365352807814
Middle East
DNE 60% ***** 0.009972233408894098
DNE 70% ***** 0.008291312828128141
DNE 80% ***** 0.0011872106803187447
DNE 90% ***** 0.003238358830955154
North Africa
DNE 60% ***** 0.034009972749627825
DNE 70% ***** 0.14642366817310887
DNE 80% ***** 0.3618720812986529
DNE 90% ***** 0.46859455784061216
North America
DNE 60% ***** 0.01358675407643012
DNE 70% ***** 0.04099662349403611
DNE 80% ***** 0.1406883180985729
DNE 90% ***** 0.01893148007463502
Oceanic
DNE 60% ***** 0.0017483948652708288
DNE 70% ***** 0.0009491598308567793
DNE 80% ***** 0.015809520056797637
DNE 90% ***** 0.021771375678482464
South/Central America
DNE 60% ***** 0.03512846737656206
DNE 70% ***** 0.034616017392936106